# Fig. S3 | Subsurface context and long-term water-table baseline

Maps of the inputs used to interpret groundwater recovery and reconstruct absolute WTD.


In [1]:
from pathlib import Path

import geopandas as gpd
import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import rasterio
from rasterio.features import geometry_mask
from matplotlib import colors, patches
from pyproj import Transformer
from shapely.geometry import LineString

ROOT = next(path.resolve() for path in [Path.cwd(), *Path.cwd().parents]
            if (path / 'outputs' / 'RECON_MAIN_2011_2023').exists())
OUT_DIR = ROOT / 'outputs' / 'figures' / 'FigS3'
OUT_DIR.mkdir(parents=True, exist_ok=True)

STREAMBED_PATH = ROOT / 'data' / '15 MRVA streambed connectiviy' / 'MRVA_streambed_connectivity_1km_aligned.tif'
CONNECTIVITY_PATH = ROOT / 'data' / '8 connectivity' / 'ConfiningLayer_SurfaceConnectivity.tif'
RESISTIVITY_PATH = ROOT / 'data' / '1 resistivity' / 'aem_log10res_1km_masked.tif'
LONGTERM_WTD_PATH = ROOT / 'data' / '6 longterm_mean_WTD' / 'longterm_wtd_1km.csv'
BOUNDARY_PATH = ROOT / 'assets' / 'spatial' / 'mrva_boundary.geojson'
RIVER_PATH = ROOT / 'assets' / 'spatial' / 'mississippi_river.gmt'

LONLAT_CRS = 'EPSG:4326'
BOUNDARY_COLOR = '#1F1F1F'
RIVER_COLOR = '#A9DCEF'
EXPORT_DPI = 600

mpl.rcParams.update({
    'font.family': 'sans-serif', 'font.sans-serif': ['Arial', 'Helvetica', 'DejaVu Sans'],
    'font.size': 9, 'axes.labelsize': 9, 'xtick.labelsize': 8, 'ytick.labelsize': 8,
    'axes.linewidth': 0.65, 'pdf.fonttype': 42, 'ps.fonttype': 42,
    'savefig.dpi': EXPORT_DPI, 'savefig.bbox': 'tight',
})


def read_raster(path: Path, band: int = 1):
    with rasterio.open(path) as src:
        values = src.read(band).astype(float)
        if src.nodata is not None:
            values[values == src.nodata] = np.nan
        boundary_in_raster_crs = boundary.to_crs(src.crs)
        inside = geometry_mask(
            [geometry.__geo_interface__ for geometry in boundary_in_raster_crs.geometry],
            out_shape=values.shape, transform=src.transform, invert=True,
        )
        values[~inside] = np.nan
        return values, src.transform, src.crs


def edge_lonlat(transform, crs, shape):
    rows, cols = np.meshgrid(np.arange(shape[0] + 1), np.arange(shape[1] + 1), indexing='ij')
    x = transform.c + transform.a * cols + transform.b * rows
    y = transform.f + transform.d * cols + transform.e * rows
    lon, lat = Transformer.from_crs(crs, LONLAT_CRS, always_xy=True).transform(x, y)
    return np.asarray(lon), np.asarray(lat)


def read_gmt(path: Path):
    segments, current = [], []
    for line in path.read_text(encoding='utf-8').splitlines():
        line = line.strip()
        if not line:
            continue
        if line.startswith('>'):
            if current:
                segments.append(np.asarray(current, dtype=float))
                current = []
        else:
            current.append(tuple(map(float, line.split()[:2])))
    if current:
        segments.append(np.asarray(current, dtype=float))
    return segments


boundary = gpd.read_file(BOUNDARY_PATH).to_crs(LONLAT_CRS)
MRVA = boundary.geometry.union_all() if hasattr(boundary.geometry, 'union_all') else boundary.unary_union
RIVER_SEGMENTS = read_gmt(RIVER_PATH)


def draw_geometry(ax, geometry, **kwargs):
    if geometry.is_empty:
        return
    if geometry.geom_type == 'LineString':
        x, y = geometry.xy
        ax.plot(x, y, **kwargs)
    elif geometry.geom_type in {'MultiLineString', 'GeometryCollection'}:
        for part in geometry.geoms:
            draw_geometry(ax, part, **kwargs)
    elif geometry.geom_type in {'Polygon', 'MultiPolygon'}:
        draw_geometry(ax, geometry.boundary, **kwargs)


def style_map(ax):
    for segment in RIVER_SEGMENTS:
        clipped = LineString(segment).intersection(MRVA)
        draw_geometry(ax, clipped, color=RIVER_COLOR, lw=0.45, zorder=3)
    draw_geometry(ax, MRVA.boundary, color=BOUNDARY_COLOR, lw=0.45, zorder=4)
    minx, miny, maxx, maxy = MRVA.bounds
    ax.set_xlim(minx, maxx)
    ax.set_ylim(miny, maxy)
    ax.set_xticks([])
    ax.set_yticks([])
    ax.tick_params(bottom=False, left=False)
    for spine in ax.spines.values():
        spine.set_visible(False)
    ax.set_aspect(1 / np.cos(np.deg2rad((miny + maxy) / 2)))


def save_continuous(values, transform, crs, name, cmap, norm, ticks, label, *, log=False):
    lon, lat = edge_lonlat(transform, crs, values.shape)
    fig, ax = plt.subplots(figsize=(3.45, 5.8), dpi=EXPORT_DPI)
    image = ax.pcolormesh(lon, lat, values, cmap=cmap, norm=norm, shading='flat', rasterized=True)
    style_map(ax)
    cbar = fig.colorbar(image, ax=ax, fraction=0.042, pad=0.025, extend='both', ticks=ticks)
    cbar.set_label(label, labelpad=6)
    cbar.ax.tick_params(length=2.5, width=0.55, labelsize=8)
    cbar.outline.set_linewidth(0.55)
    path = OUT_DIR / name
    fig.savefig(path, facecolor='white')
    plt.close(fig)
    return path


# a. Streambed vertical hydraulic conductivity, used as a proxy for streambed connectivity.
streambed, transform, crs = read_raster(STREAMBED_PATH)
paths = [save_continuous(
    streambed, transform, crs, 'FigS3_MRVA_streambed_connectivity.png',
    plt.colormaps['YlGnBu_r'], colors.LogNorm(5e-3, 15), [1e-2, 1e-1, 1, 10],
    'Streambed VIC (S)\nover = groundwater connectivity', log=True,
)]

# b. Surface confining-layer connectivity classes.
surface, transform, crs = read_raster(CONNECTIVITY_PATH)
lon, lat = edge_lonlat(transform, crs, surface.shape)
class_labels = ['Thick confining', 'Intermediate confining', 'Thin confining',
                'Some connectivity', 'Intermediate connectivity', 'High connectivity']
class_colors = ['#2025AA', '#3031D0', '#8C83DF', '#E99797', '#FF1515', '#B50000']
fig, ax = plt.subplots(figsize=(5.15, 5.8), dpi=EXPORT_DPI)
ax.pcolormesh(lon, lat, surface, cmap=colors.ListedColormap(class_colors),
              norm=colors.BoundaryNorm([-3.5, -2.5, -1.5, 0, 1.5, 2.5, 3.5], 6),
              shading='flat', rasterized=True)
style_map(ax)
handles = [patches.Patch(facecolor=color, edgecolor='none', label=label)
           for color, label in zip(class_colors, class_labels)]
ax.legend(handles=handles, loc='lower left', bbox_to_anchor=(1.01, 0.04), frameon=False, fontsize=7.0,
          handlelength=0.9, handletextpad=0.35, labelspacing=0.34, borderpad=0.15)
path = OUT_DIR / 'FigS3_MRVA_surface_connectivity.png'
fig.savefig(path, facecolor='white')
plt.close(fig)
paths.append(path)

# c. AEM log10 resistivity at 2.5 m depth.
resistivity, transform, crs = read_raster(RESISTIVITY_PATH, band=1)
paths.append(save_continuous(
    resistivity, transform, crs, 'FigS3_MRVA_surface_resistivity.png',
    plt.colormaps['RdYlBu_r'], colors.Normalize(-0.5, 3.4), [0, 1, 2, 3],
    'log$_{10}$ resistivity',
))

# d. Long-term mean WTD baseline used to recover absolute water-table depth.
longterm = pd.read_csv(LONGTERM_WTD_PATH)
wtd = np.full(resistivity.shape, np.nan, dtype=float)
wtd[longterm['row'].to_numpy(int), longterm['col'].to_numpy(int)] = longterm['wtd_model_mean_m'].to_numpy(float)
wtd = np.flipud(wtd)  # row indices increase northward; GeoTIFF rows increase southward.
wtd_limits = np.nanquantile(wtd, [0.01, 0.99])
paths.append(save_continuous(
    wtd, transform, crs, 'FigS3_MRVA_longterm_mean_WTD.png',
    plt.colormaps['YlGnBu'], colors.Normalize(*wtd_limits), [0, 5, 10, 15, 20, 25],
    'Long-term mean WTD (m)',
))

print('Saved:')
for path in paths:
    print(path.relative_to(ROOT))


Saved:
outputs\figures\FigS3\FigS3_MRVA_streambed_connectivity.png
outputs\figures\FigS3\FigS3_MRVA_surface_connectivity.png
outputs\figures\FigS3\FigS3_MRVA_surface_resistivity.png
outputs\figures\FigS3\FigS3_MRVA_longterm_mean_WTD.png
